# Ponimator: Interactive Two-Person Pose Estimation & Motion Analysis
## Multi-Environment Demo (Google Colab & Local)

[![arXiv](https://img.shields.io/badge/arXiv-2510.14976-b31b1b.svg)](https://arxiv.org/abs/2510.14976) 
[![Project Page](https://img.shields.io/badge/Project-Website-blue?style=flat&logo=Google%20chrome&logoColor=blue)](https://stevenlsw.github.io/ponimator/)

This notebook extracts interactive two-person motion data from dance videos and exports to JSON format.

**Environment Support:**
- ☁️ **Google Colab**: Requires GPU runtime (`Runtime > Change runtime type > GPU`)
- 💻 **Local Ubuntu**: Requires NVIDIA GPU with CUDA support

**Focus:** Two-person dance pose estimation → JSON export (no visualization, no generation)

## 1. Check GPU Availability

In [ ]:
import torch
import sys
import os

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
environment = "Google Colab" if IN_COLAB else "Local Machine"

print(f"🖥️  Environment: {environment}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"\n{'='*60}")

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA version: {torch.version.cuda}")
    capability = torch.cuda.get_device_capability(0)
    print(f"   GPU Compute Capability: {capability[0]}.{capability[1]}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected!")
    if IN_COLAB:
        print("\n📝 To enable GPU in Google Colab:")
        print("   1. Go to Runtime > Change runtime type")
        print("   2. Select 'T4 GPU' or 'A100 GPU' from Hardware accelerator")
        print("   3. Click Save and wait for runtime to restart")
    else:
        print("\n📝 For local Ubuntu machine:")
        print("   1. Verify NVIDIA GPU is installed: nvidia-smi")
        print("   2. Check CUDA installation: nvcc --version")

print(f"{'='*60}\n")

## 2. Mount Google Drive (Colab Only)

In [ ]:
# Mount Google Drive (Colab only)
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted at /content/drive")
else:
    print("ℹ️  Skipping Google Drive mount (not in Colab environment)")
    print("   For local usage, use absolute paths to your video files")

## 3. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository (Colab only)
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("☁️  Cloning repository for Colab environment...")
    !git clone https://github.com/stevenlsw/ponimator.git
    %cd ponimator
    print("✅ Repository cloned")
else:
    print("💻 Running on local machine - skipping repository clone")
    print("   Assuming we're already in the ponimator directory")
    print(f"   Current directory: {os.getcwd()}")
    
    # Verify we're in the right directory
    if not os.path.exists('ponimator') or not os.path.exists('scripts'):
        print("⚠️  Warning: Expected files not found. Make sure you're in the ponimator directory.")
    else:
        print("✅ Repository directory verified")

In [ ]:
# Install PyTorch 2.9+ and dependencies
print("📦 Installing PyTorch 2.9+ with CUDA 12.1...")

# Install PyTorch 2.9+
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install other dependencies (including aitviewer for SMPL layer)
!pip install -q yacs roma einops scipy scikit-learn aitviewer

print("✅ Dependencies installed")

# Verify installations
import torch
import numpy as np
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 4. Download Model Checkpoints

In [ ]:
# Download checkpoints based on environment
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

os.makedirs('checkpoints', exist_ok=True)

# Only need contactmotion checkpoint for pose estimation
checkpoint_files = {
    'contactmotion.ckpt': 'https://huggingface.co/shaoweiliu/ponimator/resolve/main/contactmotion.ckpt',
}

for filename, url in checkpoint_files.items():
    checkpoint_path = f'checkpoints/{filename}'
    
    if os.path.exists(checkpoint_path):
        size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
        print(f"✅ {filename} already exists ({size_mb:.1f} MB)")
    else:
        if IN_COLAB:
            # Try copying from Google Drive first
            drive_path = f'/content/drive/MyDrive/ponimator/{filename}'
            if os.path.exists(drive_path):
                print(f"📥 Copying {filename} from Google Drive...")
                shutil.copy2(drive_path, checkpoint_path)
                size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                print(f"✅ {filename} copied ({size_mb:.1f} MB)")
            else:
                print(f"📥 Downloading {filename} from HuggingFace...")
                !wget -q {url} -O {checkpoint_path}
                if os.path.exists(checkpoint_path):
                    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                    print(f"✅ {filename} downloaded ({size_mb:.1f} MB)")
        else:
            print(f"📥 Downloading {filename} from HuggingFace...")
            !wget -q {url} -O {checkpoint_path}
            if os.path.exists(checkpoint_path):
                size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
                print(f"✅ {filename} downloaded ({size_mb:.1f} MB)")

print("\n✅ Checkpoint ready!")

## 5. Setup SMPL-X Body Models

In [ ]:
# Setup SMPL-X models based on environment
import os
import sys
import shutil

IN_COLAB = 'google.colab' in sys.modules

os.makedirs('body_models/smplx', exist_ok=True)

required_files = {
    'SMPLX_MALE.npz': 'SMPL-X male model',
    'SMPLX_FEMALE.npz': 'SMPL-X female model',
    'SMPLX_NEUTRAL.npz': 'SMPL-X neutral model',
    'SMPLX_MALE.pkl': 'SMPL-X male model (pkl)',
    'SMPLX_FEMALE.pkl': 'SMPL-X female model (pkl)',
    'SMPLX_NEUTRAL.pkl': 'SMPL-X neutral model (pkl)',
}

if IN_COLAB:
    # Copy from Google Drive
    base_dir = '/content/drive/MyDrive/smplx'
    
    if not os.path.exists(base_dir):
        print(f"❌ Source directory not found: {base_dir}")
        print("   Please make sure SMPL-X models are in your Google Drive at:")
        print("   /content/drive/MyDrive/smplx/")
    else:
        print(f"📁 Copying SMPL-X models from: {base_dir}")
        
        for filename, desc in required_files.items():
            source_path = os.path.join(base_dir, filename)
            dest_path = f'body_models/smplx/{filename}'
            
            if os.path.exists(source_path):
                shutil.copy2(source_path, dest_path)
                size_kb = os.path.getsize(dest_path) / 1024
                print(f"  ✓ Copied {desc}: {filename} ({size_kb:.1f} KB)")
            else:
                print(f"  ✗ Not found: {source_path}")
else:
    # Local machine - check if files exist
    print("🔍 Checking for existing SMPL-X models...")
    
    missing_files = []
    for filename, desc in required_files.items():
        file_path = f'body_models/smplx/{filename}'
        
        if os.path.exists(file_path):
            size_kb = os.path.getsize(file_path) / 1024
            print(f"  ✓ {desc}: {filename} ({size_kb:.1f} KB)")
        else:
            print(f"  ✗ {desc}: {filename} NOT FOUND")
            missing_files.append((filename, desc))
    
    if missing_files:
        print(f"\n⚠️  {len(missing_files)} file(s) missing")
        print("\nTo download missing files:")
        print("  1. Register at https://smpl-x.is.tue.mpg.de")
        print("  2. Download SMPL-X models")
        print("  3. Place files in body_models/smplx/")
    else:
        print("\n✅ All SMPL-X models found!")

## 6. Run Two-Person Pose Estimation from Dance Video

This extracts interactive poses from your dance video using Buddi pose estimation.

**Note:** You need to first run Buddi pose estimation on your video. See the README for instructions on using the custom Buddi script.

In [ ]:
# Set video path and data directory based on environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Google Colab: Use Google Drive paths
    video_path = '/content/drive/MyDrive/Zouk/3D-Pose/carlos-aline/carlos-aline-spin-cam1.mp4'
    # Buddi output directory (after running Buddi pose estimation)
    data_dir = '/content/drive/MyDrive/Zouk/3D-Pose/carlos-aline/buddi_output'
    print("☁️  Running in Google Colab")
else:
    # Local machine: Use local paths
    video_path = '/home/john/Videos/Zouk/carlos-aline-spin-cam2.mp4'
    # Buddi output directory (after running Buddi pose estimation)
    data_dir = '/home/john/Videos/Zouk/carlos-aline-spin-cam2_buddi'
    print("💻 Running on local machine")

print(f"📹 Video path: {video_path}")
print(f"📁 Buddi data directory: {data_dir}")

# Verify files exist
if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f"✅ Video found ({size_mb:.1f} MB)")
else:
    print(f"⚠️  Video not found: {video_path}")

if os.path.exists(data_dir):
    print(f"✅ Buddi data directory found")
else:
    print(f"⚠️  Buddi data not found: {data_dir}")
    print(f"   You need to run Buddi pose estimation first!")
    print(f"   See README 'Custom Third-party Scripts' section")

# Set output directory
output_dir = "outputs/carlos_aline_dance"
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

In [ ]:
# Run pose2motion inference with JSON export only (no visualization)
import torch

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✓ GPU Ready: {torch.cuda.get_device_name(0)}")

# Run inference - JSON export only, no visualization
!python scripts/run_pose2motion.py \
    --data_dir {data_dir} \
    --save_dir {output_dir} \
    --save \
    --disable_vis \
    --export_json

print(f"\n✅ Processing complete! Results saved to: {output_dir}")
print(f"   - motion_pred.pkl: Raw motion data")
print(f"   - poses.json: Two-person dance motion in JSON format")

## 7. Download Results (Colab Only)

In [ ]:
# Download JSON results
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

# Find all poses.json files
poses_files = []
for root, dirs, files in os.walk('outputs'):
    for file in files:
        if file == 'poses.json':
            poses_files.append(os.path.join(root, file))

if IN_COLAB:
    from google.colab import files
    
    for poses_json in poses_files:
        if os.path.exists(poses_json):
            size_mb = os.path.getsize(poses_json) / (1024 * 1024)
            print(f"📥 Downloading: {poses_json} ({size_mb:.2f} MB)")
            files.download(poses_json)
    
    print("\n✅ All JSON files downloaded!")
else:
    print("💻 Running on local machine")
    print("\n📁 JSON files saved at:")
    for poses_json in poses_files:
        if os.path.exists(poses_json):
            size_mb = os.path.getsize(poses_json) / (1024 * 1024)
            print(f"   {os.path.abspath(poses_json)} ({size_mb:.2f} MB)")

## Understanding the JSON Output Format

The exported `poses.json` contains two-person dance motion data:

### Metadata
- Sequence name
- **Coordinate system**: Camera space (meters)
- Format version: **1.0**
- Number of frames and persons
- Interactive frame index

### Per-Frame Data
For each frame, you get:
- **Person 0 and Person 1 parameters**:
  - `person_id`: Unique identifier (0 or 1)
  - `smplx_parameters`:
    - `betas`: 10D shape parameters
    - `root_orient`: 6D global orientation
    - `body_pose`: Body joint rotations (6D format)
    - `translation`: 3D position in camera space
    - `gender`: Person's gender (0=male, 1=female, 2=neutral)

### Example: Load and Analyze Dance Motion

```python
import json
import numpy as np

# Load dance poses
with open('outputs/carlos_aline_dance/poses.json', 'r') as f:
    data = json.load(f)

# Get metadata
metadata = data['metadata']
print(f"Sequence: {metadata['sequence_name']}")
print(f"Total frames: {metadata['total_frames']}")
print(f"Number of dancers: {metadata['num_persons']}")

# Get first frame
frame_0 = data['frames']['0']
person_0 = frame_0['persons'][0]
person_1 = frame_0['persons'][1]

print(f"\nPerson 0 position: {person_0['smplx_parameters']['translation']}")
print(f"Person 1 position: {person_1['smplx_parameters']['translation']}")

# Calculate distance between dancers
pos_0 = np.array(person_0['smplx_parameters']['translation'])
pos_1 = np.array(person_1['smplx_parameters']['translation'])
distance = np.linalg.norm(pos_1 - pos_0)
print(f"Distance between dancers: {distance:.2f} meters")
```

### What This JSON Contains

- ✅ Full SMPL-X body parameters for both dancers
- ✅ 3D positions and orientations for every frame
- ✅ Shape parameters (body proportions)
- ✅ All data in camera coordinate space
- ✅ Ready for import into Unity, Blender, or custom analysis tools

---

## Next Steps

### 1. Run Buddi Pose Estimation First

Before using this notebook, you need to extract two-person poses from your video using Buddi:

```bash
# See ponimator README for custom Buddi script
cd buddi/
./custom_demo.sh /path/to/your/dance_video.mp4 0
```

### 2. Process the Data

Once you have Buddi output, run this notebook to:
- Load the interactive poses
- Generate smooth motion sequences
- Export to JSON format

### 3. Use the JSON Output

Import the JSON into your application:
- Unity for real-time rendering
- Blender for animation
- Python for motion analysis
- Any tool that reads JSON

---

## Citation

If you find this work useful, please cite:

```bibtex
@inproceedings{liu2025ponimator,
    title={Ponimator: Unfolding Interactive Pose for Versatile Human-Human Interaction Animation},
    author={Liu, Shaowei and Guo, Chuan and Zhou, Bing and Wang, Jian},
    booktitle={International Conference on Computer Vision (ICCV)},
    year={2025}
}
```